# Multi-output XGBoost

Code is based off [xgboostWB_electricity.py](https://github.com/Daniela-Shereen/GBRT-for-TSF/blob/main/XGBoost_(W-b)/Univariate/xgboostWB_electricity.py) example from research paper "Do We Really Need Deep Learning Models for Time Series Forecasting?" with modifications for the research dataset.

## Set-Up

In [1]:
import numpy as np 
import math
import pandas as pd 
import xgboost as xgb
from sklearn.multioutput import MultiOutputRegressor
import os, pathlib

Following code to load in data is pulled from Jeff's EVL file.

In [2]:
# >>> If you used the exact folder name you shared:
BASE = "../data/NSW/processed"

# Prefer parquet (smaller/faster); fall back to csv
PARQ = f"{BASE}/nsw_demand_features.parquet"
CSV  = f"{BASE}/nsw_demand_features.csv"
DATA_PATH = PARQ if pathlib.Path(PARQ).exists() else CSV
assert pathlib.Path(DATA_PATH).exists(), f"File not found at {DATA_PATH}"

# Env + split years per team slides
os.environ["DATA_PATH"]      = DATA_PATH
os.environ["VAL_YEAR"]       = "2018"
os.environ["TEST_YEAR"]      = "2019"
os.environ["TEST_END_YEAR"]  = "2021"   # "" to use all remaining years
os.environ["USE_EXOG"]       = "0"      # 0 = history-only drop; 1 = keep exogenous (temp, etc.)

print("Using:", os.environ["DATA_PATH"])

Using: ../data/NSW/processed/nsw_demand_features.parquet


In [3]:
VAL_YEAR, TEST_YEAR, TEST_END_YEAR = 2018, 2019, 2021
USE_EXOG = 0
OUT_DIR = pathlib.Path("outputs/xgboost"); OUT_DIR.mkdir(parents=True, exist_ok=True)

p = os.environ.get("DATA_PATH")
assert p and pathlib.Path(p).exists(), f"DATA_PATH missing or not found: {p}"
df = pd.read_parquet(p) if p.endswith((".parquet",".pq")) else pd.read_csv(p)

needed = {"date","hour","y"}
missing = needed - set(df.columns)
assert not missing, f"Missing required columns: {missing}"

df = df.copy()
df["date"] = pd.to_datetime(df["date"], errors="coerce")
mins = np.rint(df["hour"].astype(float)*60).astype(int)
df["ds"] = df["date"] + pd.to_timedelta(mins, unit="m")
df = df.sort_values("ds").reset_index(drop=True)
df["year"] = df["ds"].dt.year

def drop_exog(d):
    if USE_EXOG: return d
    drop_like = ["forecast"]
    keep = [c for c in d.columns if not any(k in c.lower() for k in drop_like)]
    return d[keep]
df = drop_exog(df)

## Pre-processing

In [6]:
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import Normalizer
#from datetime import datetime, timedelta
from sklearn import preprocessing

Get a summary of the current state of the data.

In [7]:
print(df.shape)
print(df.columns.values)
df.describe()

(196513, 31)
['temp' 'y' 'hour' 'dow' 'is_weekend' 'sin_hh' 'cos_hh' 'sin_dow'
 'cos_dow' 'is_holiday' 'date' 'is_long_weekend' 'is_hot_slot'
 'is_cold_slot' 'is_hot_day' 'is_cold_day' 'is_normal_day' 'y_lag1'
 'y_lag48' 'y_lag336' 'y_roll_mean_48' 'y_roll_std_48' 'y_roll_mean_336'
 'temp_lag1' 'temp_lag2' 'temp_lag3' 'temp_lag48' 'temp_roll_max_48'
 'temp_roll_min_48' 'ds' 'year']


,temp,y,hour,dow,is_weekend,sin_hh,cos_hh,sin_dow,cos_dow,is_holiday,...,y_roll_std_48,y_roll_mean_336,temp_lag1,temp_lag2,temp_lag3,temp_lag48,temp_roll_max_48,temp_roll_min_48,ds,year
count,196513.000000,196513.000000,196513.00000,196513.000000,196513.000000,1.965130e+05,1.965130e+05,196513.000000,196513.000000,196513.000000,...,196489.000000,196417.000000,196512.000000,196511.000000,196510.000000,196465.000000,196489.000000,196489.000000,196513,196513.000000
mean,17.526938,8113.145859,11.74994,3.000000,0.285783,-9.102662e-18,5.088722e-06,-0.000104,0.000215,0.033219,...,1001.296115,8113.703673,17.526933,17.526926,17.526918,17.526420,23.129053,12.602414,2015-08-10 00:00:00.000000512,2015.112145
min,-1.300000,5074.630000,0.00000,0.000000,0.000000,-1.000000e+00,-1.000000e+00,-0.974928,-0.900969,0.000000,...,353.881138,6578.555298,-1.300000,-1.300000,-1.300000,-1.300000,11.700000,-1.300000,2010-01-01 00:00:00,2010.000000
25%,13.500000,7150.070000,5.50000,1.000000,0.000000,-7.071068e-01,-7.071068e-01,-0.781831,-0.900969,0.000000,...,801.600415,7585.017560,13.500000,13.500000,13.500000,13.500000,19.150000,8.200000,2012-10-20 12:00:00,2012.000000
50%,17.850000,8053.230000,11.50000,3.000000,0.000000,0.000000e+00,6.123234e-17,0.000000,-0.222521,0.000000,...,971.987245,8080.426518,17.850000,17.850000,17.850000,17.850000,22.700000,12.800000,2015-08-10 00:00:00,2015.000000
75%,21.500000,8958.550000,17.50000,5.000000,1.000000,7.071068e-01,7.071068e-01,0.781831,0.623490,0.000000,...,1153.577956,8558.597440,21.500000,21.500000,21.500000,21.500000,26.500000,17.100000,2018-05-29 12:00:00,2018.000000
max,44.700000,14579.860000,23.50000,6.000000,1.000000,1.000000e+00,1.000000e+00,0.974928,1.000000,1.000000,...,2703.844474,10901.113512,44.700000,44.700000,44.700000,44.700000,44.700000,26.600000,2021-03-18 00:00:00,2021.000000
std,5.881148,1299.532774,6.92675,2.000244,0.451788,7.071068e-01,7.071104e-01,0.707161,0.707056,0.179209,...,277.513273,653.871791,5.881163,5.881177,5.881191,5.881735,5.107634,5.358207,NaN,3.235967


In [8]:
df.set_index('ds', inplace=True) # set the timestamp as the index
df = df.drop(columns = ['date', 'is_hot_slot', 'is_cold_slot', 'is_hot_day', 'is_cold_day', 'is_normal_day', 'temp'])

In [9]:
print(df.shape)
print(df.columns.values)
df.describe()

(196513, 23)
['y' 'hour' 'dow' 'is_weekend' 'sin_hh' 'cos_hh' 'sin_dow' 'cos_dow'
 'is_holiday' 'is_long_weekend' 'y_lag1' 'y_lag48' 'y_lag336'
 'y_roll_mean_48' 'y_roll_std_48' 'y_roll_mean_336' 'temp_lag1'
 'temp_lag2' 'temp_lag3' 'temp_lag48' 'temp_roll_max_48'
 'temp_roll_min_48' 'year']


,y,hour,dow,is_weekend,sin_hh,cos_hh,sin_dow,cos_dow,is_holiday,is_long_weekend,...,y_roll_mean_48,y_roll_std_48,y_roll_mean_336,temp_lag1,temp_lag2,temp_lag3,temp_lag48,temp_roll_max_48,temp_roll_min_48,year
count,196513.000000,196513.00000,196513.000000,196513.000000,1.965130e+05,1.965130e+05,196513.000000,196513.000000,196513.000000,196513.000000,...,196489.000000,196489.000000,196417.000000,196512.000000,196511.000000,196510.000000,196465.000000,196489.000000,196489.000000,196513.000000
mean,8113.145859,11.74994,3.000000,0.285783,-9.102662e-18,5.088722e-06,-0.000104,0.000215,0.033219,0.044455,...,8113.214173,1001.296115,8113.703673,17.526933,17.526926,17.526918,17.526420,23.129053,12.602414,2015.112145
std,1299.532774,6.92675,2.000244,0.451788,7.071068e-01,7.071104e-01,0.707161,0.707056,0.179209,0.206104,...,794.857408,277.513273,653.871791,5.881163,5.881177,5.881191,5.881735,5.107634,5.358207,3.235967
min,5074.630000,0.00000,0.000000,0.000000,-1.000000e+00,-1.000000e+00,-0.974928,-0.900969,0.000000,0.000000,...,5944.900625,353.881138,6578.555298,-1.300000,-1.300000,-1.300000,-1.300000,11.700000,-1.300000,2010.000000
25%,7150.070000,5.50000,1.000000,0.000000,-7.071068e-01,-7.071068e-01,-0.781831,-0.900969,0.000000,0.000000,...,7554.742500,801.600415,7585.017560,13.500000,13.500000,13.500000,13.500000,19.150000,8.200000,2012.000000
50%,8053.230000,11.50000,3.000000,0.000000,0.000000e+00,6.123234e-17,0.000000,-0.222521,0.000000,0.000000,...,8063.048958,971.987245,8080.426518,17.850000,17.850000,17.850000,17.850000,22.700000,12.800000,2015.000000
75%,8958.550000,17.50000,5.000000,1.000000,7.071068e-01,7.071068e-01,0.781831,0.623490,0.000000,0.000000,...,8643.396250,1153.577956,8558.597440,21.500000,21.500000,21.500000,21.500000,26.500000,17.100000,2018.000000
max,14579.860000,23.50000,6.000000,1.000000,1.000000e+00,1.000000e+00,0.974928,1.000000,1.000000,1.000000,...,11715.658125,2703.844474,10901.113512,44.700000,44.700000,44.700000,44.700000,44.700000,26.600000,2021.000000


Following code is heavily based on [xgboostWB_electricity.py](https://github.com/Daniela-Shereen/GBRT-for-TSF/blob/main/XGBoost_(W-b)/Univariate/xgboostWB_electricity.py) from research paper "Do We Really Need Deep Learning Models for Time Series Forecasting?", adapted to the dataset for this project.

In [10]:
# apply min_max scaling to the lagged temp and demand features
# to be from -0.5 to 0.5 as mentioned in the paper
normalise_columns = ['y_lag1', 'y_lag48', 'y_lag336',\
                     'y_roll_mean_48', 'y_roll_std_48', 'y_roll_mean_336', \
                     'temp_lag1', 'temp_lag2', 'temp_lag3', 'temp_lag48', \
                     'temp_roll_max_48', 'temp_roll_min_48']

for colname in normalise_columns:
    df[colname] = MinMaxScaler(feature_range=(-0.5, 0.5)).fit_transform(np.array(df[colname]).reshape(-1,1))

In [11]:
df.describe()

,y,hour,dow,is_weekend,sin_hh,cos_hh,sin_dow,cos_dow,is_holiday,is_long_weekend,...,y_roll_mean_48,y_roll_std_48,y_roll_mean_336,temp_lag1,temp_lag2,temp_lag3,temp_lag48,temp_roll_max_48,temp_roll_min_48,year
count,196513.000000,196513.00000,196513.000000,196513.000000,1.965130e+05,1.965130e+05,196513.000000,196513.000000,196513.000000,196513.000000,...,196489.000000,196489.000000,196417.000000,196512.000000,196511.000000,196510.000000,196465.000000,196489.000000,196489.000000,196513.000000
mean,8113.145859,11.74994,3.000000,0.285783,-9.102662e-18,5.088722e-06,-0.000104,0.000215,0.033219,0.044455,...,-0.124258,-0.224500,-0.144852,-0.090719,-0.090719,-0.090719,-0.090730,-0.153665,-0.001706,2015.112145
std,1299.532774,6.92675,2.000244,0.451788,7.071068e-01,7.071104e-01,0.707161,0.707056,0.179209,0.206104,...,0.137739,0.118093,0.151270,0.127851,0.127852,0.127852,0.127864,0.154777,0.192050,3.235967
min,5074.630000,0.00000,0.000000,0.000000,-1.000000e+00,-1.000000e+00,-0.974928,-0.900969,0.000000,0.000000,...,-0.500000,-0.500000,-0.500000,-0.500000,-0.500000,-0.500000,-0.500000,-0.500000,-0.500000,2010.000000
25%,7150.070000,5.50000,1.000000,0.000000,-7.071068e-01,-7.071068e-01,-0.781831,-0.900969,0.000000,0.000000,...,-0.221035,-0.309478,-0.267161,-0.178261,-0.178261,-0.178261,-0.178261,-0.274242,-0.159498,2012.000000
50%,8053.230000,11.50000,3.000000,0.000000,0.000000e+00,6.123234e-17,0.000000,-0.222521,0.000000,0.000000,...,-0.132951,-0.236972,-0.152550,-0.083696,-0.083696,-0.083696,-0.083696,-0.166667,0.005376,2015.000000
75%,8958.550000,17.50000,5.000000,1.000000,7.071068e-01,7.071068e-01,0.781831,0.623490,0.000000,0.000000,...,-0.032385,-0.159698,-0.041928,-0.004348,-0.004348,-0.004348,-0.004348,-0.051515,0.159498,2018.000000
max,14579.860000,23.50000,6.000000,1.000000,1.000000e+00,1.000000e+00,0.974928,1.000000,1.000000,1.000000,...,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,0.500000,2021.000000


In [65]:
num_periods_output = 48 #to predict
num_periods_input = 48 #input

In [66]:
def New_preprocessing(Data_df):
    Number_Of_Features = len(Data_df.columns)
    ############################################ Windowing ##################################
    end = len(Data_df)
    start = 0
    next = 0
    x_batches = []
    y_batches = []  
    count = 0
    while (next+num_periods_input) < end:
        if start % 10000 == 0: print("row ", start)
        next = start+num_periods_input
        #print(start, next)
        x_batches.append(Data_df.iloc[start:next,:])
        y_batches.append(Data_df.iloc[next:next+num_periods_output,0]) # first col is y
        start = start+1
    y_batches = np.asarray(y_batches)
    y_batches = y_batches.reshape(-1, num_periods_output, 1) 
    print('Length of y batches :',len(y_batches),' ',num_periods_input,' ',num_periods_output)
    #print(x_batches)
    x_batches=np.asarray(x_batches) 
    x_batches = x_batches.reshape(-1, num_periods_input, Number_Of_Features)
    print('len x_batches ',len(x_batches))

    return x_batches, y_batches

In [67]:
###################################################
x_batches_Full=[]
y_batches_Full=[]
x_batches_Full, y_batches_Full = New_preprocessing(df)

row  0
row  10000
row  20000
row  30000
row  40000
row  50000
row  60000
row  70000
row  80000
row  90000
row  100000
row  110000
row  120000
row  130000
row  140000
row  150000
row  160000
row  170000
row  180000
row  190000
Length of y batches : 196418   48   48
len x_batches  196418


In [68]:
# check format of data
print(x_batches_Full.shape)
print(y_batches_Full.shape)

(196418, 48, 23)
(196418, 48, 1)


In [69]:
# reformat the training instances
All_Training_Instances=[]
 
#=============== flatten each training window into Instance =================================
for i in range(0,len(x_batches_Full)):
    hold=[]
    for j in range(0,len(x_batches_Full[i])):
    #**************** to run without features -->comment if else condition (just keep else statement) **************************
      if j==(len(x_batches_Full[i])-1):
          hold=np.concatenate((hold, x_batches_Full[i][j][:]), axis=None)
          
      else:
          hold=np.concatenate((hold, x_batches_Full[i][j][0]), axis=None)
          
    All_Training_Instances.append(hold)
    

print(len(All_Training_Instances[0]))

70


In [70]:
# change structure of training data
All_Training_Instances=np.reshape(All_Training_Instances, (len(All_Training_Instances), len(All_Training_Instances[0])))
All_Training_y = np.reshape(y_batches_Full, (len(y_batches_Full),num_periods_output))

In [71]:
# check resulting data
print(All_Training_Instances[0])
print(All_Training_Instances.shape)

[ 8.03800000e+03  7.80931000e+03  7.48369000e+03  7.11723000e+03
  6.81203000e+03  6.54433000e+03  6.37732000e+03  6.28285000e+03
  6.21149000e+03  6.24831000e+03  6.19861000e+03  6.23735000e+03
  6.37048000e+03  6.50642000e+03  6.73875000e+03  6.98041000e+03
  7.21333000e+03  7.51560000e+03  7.76146000e+03  7.91554000e+03
  8.06716000e+03  8.12331000e+03  8.19551000e+03  8.22136000e+03
  8.27809000e+03  8.36158000e+03  8.33792000e+03  8.42108000e+03
  8.47734000e+03  8.50139000e+03  8.55794000e+03  8.60311000e+03
  8.71569000e+03  8.79161000e+03  8.88398000e+03  8.83592000e+03
  8.76546000e+03  8.64056000e+03  8.54455000e+03  8.56212000e+03
  8.63578000e+03  8.57305000e+03  8.33354000e+03  8.34380000e+03
  8.29559000e+03  8.21054000e+03  8.04177000e+03  7.78268000e+03
  2.35000000e+01  4.00000000e+00  0.00000000e+00 -1.30526192e-01
  9.91444861e-01 -4.33883739e-01 -9.00968868e-01  1.00000000e+00
  1.00000000e+00 -1.87841325e-01             nan             nan
 -1.78231222e-01 -2.71606

In [77]:
# clean feature and target training data
Training_x = All_Training_Instances[All_Training_Instances[:,-1]<VAL_YEAR]
Training_x = np.delete(Training_x, np.s_[-1], axis=1) # remove year (last column)
Training_y = All_Training_y[0:Training_x.shape[0],:] # match number of rows with the x
# check format of resulting data
print(Training_x.shape)
print(Training_y.shape)
print(Training_x[0])
print(Training_y[0])
# last value in the demand window should match the first value in the test data
print('Check match: ', Training_x[1][num_periods_input-1], Training_y[0][0]) 

(140209, 69)
(140209, 48)
[ 8.03800000e+03  7.80931000e+03  7.48369000e+03  7.11723000e+03
  6.81203000e+03  6.54433000e+03  6.37732000e+03  6.28285000e+03
  6.21149000e+03  6.24831000e+03  6.19861000e+03  6.23735000e+03
  6.37048000e+03  6.50642000e+03  6.73875000e+03  6.98041000e+03
  7.21333000e+03  7.51560000e+03  7.76146000e+03  7.91554000e+03
  8.06716000e+03  8.12331000e+03  8.19551000e+03  8.22136000e+03
  8.27809000e+03  8.36158000e+03  8.33792000e+03  8.42108000e+03
  8.47734000e+03  8.50139000e+03  8.55794000e+03  8.60311000e+03
  8.71569000e+03  8.79161000e+03  8.88398000e+03  8.83592000e+03
  8.76546000e+03  8.64056000e+03  8.54455000e+03  8.56212000e+03
  8.63578000e+03  8.57305000e+03  8.33354000e+03  8.34380000e+03
  8.29559000e+03  8.21054000e+03  8.04177000e+03  7.78268000e+03
  2.35000000e+01  4.00000000e+00  0.00000000e+00 -1.30526192e-01
  9.91444861e-01 -4.33883739e-01 -9.00968868e-01  1.00000000e+00
  1.00000000e+00 -1.87841325e-01             nan             nan

In [74]:
#=========================== CALLING XGBOOST ===========================
model=xgb.XGBRegressor(learning_rate =0.2,
 n_estimators=20,
 max_depth=8,
 min_child_weight=1,
 gamma=0.0,
 subsample=0.8,
 colsample_bytree=0.8,
 scale_pos_weight=1,
 seed=42)

multioutput=MultiOutputRegressor(model).fit(Training_x,Training_y)

print('Fitting Done!')


Fitting Done!


In [78]:
# clean feature and target test data
# get features
# shift testing timeslots so that we can get 24hr-ahead forecast for first test slot
Test_x = All_Training_Instances[All_Training_Instances[:,-1]>=TEST_YEAR]
Test_x = np.delete(Test_x, np.s_[-1], axis=1) # remove year (last column)
# keep record of the timestamps for the tests (to join on to cohort evaluation)
Test_time = df.iloc[list(All_Training_Instances[:,-1]>=TEST_YEAR) + [False for i in range(num_periods_input+num_periods_output-1)]].index.values
# get target values
Test_y = All_Training_y[-Test_x.shape[0]:,:]
# check format of resulting data
print(Test_x.shape)
print(Test_time.shape)
print(Test_y.shape)
print(Test_x[0]) 
print(Test_time[0])
print(Test_y[0])
# last value in the demand window should match the first value in the test data
print('Check match: ', Test_x[1][num_periods_input-1], Test_y[0][0]) 

(38689, 69)
(38689,)
(38689, 48)
[ 7.28024000e+03  7.13330000e+03  6.86545000e+03  6.69899000e+03
  6.55732000e+03  6.52164000e+03  6.51798000e+03  6.47792000e+03
  6.57835000e+03  6.63121000e+03  6.83735000e+03  7.06518000e+03
  7.27911000e+03  7.58588000e+03  7.84752000e+03  8.11781000e+03
  8.46378000e+03  8.65278000e+03  8.74907000e+03  8.89089000e+03
  9.00586000e+03  9.27925000e+03  9.49363000e+03  9.73098000e+03
  9.96804000e+03  1.01596800e+04  1.02659400e+04  1.03405500e+04
  1.05521400e+04  1.06268200e+04  1.06505400e+04  1.05374100e+04
  1.03363700e+04  1.01147900e+04  9.73404000e+03  9.35002000e+03
  8.95753000e+03  8.71712000e+03  8.49908000e+03  8.37198000e+03
  8.15107000e+03  8.05932000e+03  8.13832000e+03  8.05374000e+03
  8.04391000e+03  7.90173000e+03  7.72044000e+03  7.61274000e+03
  0.00000000e+00  1.00000000e+00  0.00000000e+00  0.00000000e+00
  1.00000000e+00  7.81831482e-01  6.23489802e-01  1.00000000e+00
  1.00000000e+00 -2.21646925e-01 -2.43680058e-01 -3.12322

In [79]:
#============================== PREDICTION ===============================
prediction=multioutput.predict(Test_x)
print('prediction ',prediction.shape)
print('test ',Test_y.shape)

prediction  (38689, 48)
test  (38689, 48)


In [80]:
def mean_absolute_percentage_error(y_true, y_pred): 
    a=(y_true - y_pred)
    b=y_true
    c=np.divide(a, b, out=np.zeros_like(a), where=b!=0)
    return np.mean(np.abs(c)) * 100

In [81]:
MSE=np.mean((prediction - Test_y)**2)
MAE=np.mean(np.abs((prediction - Test_y)))
MAPE=mean_absolute_percentage_error(Test_y, prediction)
Bias = np.mean(prediction - Test_y)
#print('With Features for {} weeks'.format(No_Of_weeks)) 
print('MAPE: ',MAPE)
print('MAE: ',MAE)
#print('With Features for {} weeks'.format(No_Of_weeks)) 
print('RMSE: ',MSE**0.5)
print('Bias: ',Bias)

MAPE:  3.954811507767907
MAE:  311.8526038920935
RMSE:  477.57936879128795
Bias:  62.93438430531753


In [84]:
colname_pred_prefix = "tplus"
prediction_df = pd.DataFrame(prediction, columns = [colname_pred_prefix+str(i) for i in range(num_periods_output)])
print(prediction_df.shape)
prediction_df['timestamp'] = pd.Series(Test_time)
print(prediction_df.shape)

(38689, 48)
(38689, 49)


In [91]:
# re-arrange data so that all the predictions for the same timestamp are gathered in the same row
colname_rearrange_prefix = "tminus"
for i in range(num_periods_output):
    prediction_df[colname_rearrange_prefix+str(i)] = prediction_df[colname_pred_prefix+str(i)].shift(i)

print(prediction_df.shape)
print(prediction_df.columns.values)

(38689, 97)
['tplus0' 'tplus1' 'tplus2' 'tplus3' 'tplus4' 'tplus5' 'tplus6' 'tplus7'
 'tplus8' 'tplus9' 'tplus10' 'tplus11' 'tplus12' 'tplus13' 'tplus14'
 'tplus15' 'tplus16' 'tplus17' 'tplus18' 'tplus19' 'tplus20' 'tplus21'
 'tplus22' 'tplus23' 'tplus24' 'tplus25' 'tplus26' 'tplus27' 'tplus28'
 'tplus29' 'tplus30' 'tplus31' 'tplus32' 'tplus33' 'tplus34' 'tplus35'
 'tplus36' 'tplus37' 'tplus38' 'tplus39' 'tplus40' 'tplus41' 'tplus42'
 'tplus43' 'tplus44' 'tplus45' 'tplus46' 'tplus47' 'timestamp' 'tminus0'
 'tminus1' 'tminus2' 'tminus3' 'tminus4' 'tminus5' 'tminus6' 'tminus7'
 'tminus8' 'tminus9' 'tminus10' 'tminus11' 'tminus12' 'tminus13'
 'tminus14' 'tminus15' 'tminus16' 'tminus17' 'tminus18' 'tminus19'
 'tminus20' 'tminus21' 'tminus22' 'tminus23' 'tminus24' 'tminus25'
 'tminus26' 'tminus27' 'tminus28' 'tminus29' 'tminus30' 'tminus31'
 'tminus32' 'tminus33' 'tminus34' 'tminus35' 'tminus36' 'tminus37'
 'tminus38' 'tminus39' 'tminus40' 'tminus41' 'tminus42' 'tminus43'
 'tminus44' 'tminu

# Evaluation on test cohort

In [149]:
COHORT_OVERALL_ALL  = f"{BASE}/cohort3_overall_all.csv"
assert pathlib.Path(COHORT_OVERALL_ALL).exists(), f"File not found at {COHORT_OVERALL_ALL}"
df_cohort_overall_all = pd.read_csv(COHORT_OVERALL_ALL)
print(df_cohort_overall_all)

COHORT_HOTDAY_ALL  = f"{BASE}/cohort3_hotday_all.csv"
assert pathlib.Path(COHORT_HOTDAY_ALL).exists(), f"File not found at {COHORT_HOTDAY_ALL}"
df_cohort_hotday_all = pd.read_csv(COHORT_HOTDAY_ALL)
print(df_cohort_hotday_all)

                 timestamp  is_hot_day     true  op_latest   op_24h  \
0      2019-01-01 00:00:00           1  7612.74    7662.71  7734.27   
1      2019-01-01 00:30:00           1  7457.58    7445.34  7471.96   
2      2019-01-01 01:00:00           1  7243.21    7244.05  7204.11   
3      2019-01-01 01:30:00           1  6918.55    6992.60  6854.39   
4      2019-01-01 02:00:00           1  6676.58    6700.60  6598.30   
...                    ...         ...      ...        ...      ...   
38731  2021-03-17 21:30:00           0  7503.12    7518.59  7364.36   
38732  2021-03-17 22:00:00           0  7419.77    7409.33  7284.49   
38733  2021-03-17 22:30:00           0  7417.91    7422.63  7240.18   
38734  2021-03-17 23:00:00           0  7287.32    7313.13  7145.45   
38735  2021-03-17 23:30:00           0  7172.39    7192.94  7011.96   

       op_24h_latest  
0            7734.27  
1            7471.96  
2            7204.11  
3            6854.39  
4            6598.30  
...      

In [103]:
drop_like = [colname_pred_prefix]
keep = [c for c in prediction_df.columns if not any(k in c.lower() for k in drop_like)]
predicted_values_df = prediction_df[keep]
print(predicted_values_df.shape)
print(predicted_values_df.columns.values)

(38689, 49)
['timestamp' 'tminus0' 'tminus1' 'tminus2' 'tminus3' 'tminus4' 'tminus5'
 'tminus6' 'tminus7' 'tminus8' 'tminus9' 'tminus10' 'tminus11' 'tminus12'
 'tminus13' 'tminus14' 'tminus15' 'tminus16' 'tminus17' 'tminus18'
 'tminus19' 'tminus20' 'tminus21' 'tminus22' 'tminus23' 'tminus24'
 'tminus25' 'tminus26' 'tminus27' 'tminus28' 'tminus29' 'tminus30'
 'tminus31' 'tminus32' 'tminus33' 'tminus34' 'tminus35' 'tminus36'
 'tminus37' 'tminus38' 'tminus39' 'tminus40' 'tminus41' 'tminus42'
 'tminus43' 'tminus44' 'tminus45' 'tminus46' 'tminus47']


In [145]:
predicted_values_pivot = pd.wide_to_long(predicted_values_df, stubnames='tminus', i=['timestamp'], j='slot')
predicted_values_pivot.reset_index(inplace = True)
predicted_values_pivot.rename(columns={'tminus': 'xgb_pred'}, inplace=True)
print(predicted_values_pivot.shape)
print(predicted_values_pivot.head())

(1857072, 3)
            timestamp  slot     xgb_pred
0 2018-12-31 00:30:00     0  7368.561035
1 2018-12-31 01:00:00     0  7247.174805
2 2018-12-31 01:30:00     0  6978.793457
3 2018-12-31 02:00:00     0  6740.884766
4 2018-12-31 02:30:00     0  6541.900879


In [146]:
df_cohort_overall_eval = df_cohort_overall_all.copy()
df_cohort_overall_eval['timestamp'] = pd.to_datetime(df_cohort_overall_eval['timestamp'])
df_cohort_overall_eval = pd.merge(df_cohort_overall_eval, predicted_values_pivot, how='left')
#print(df_cohort_overall_eval.columns.values)

prediction = df_cohort_overall_eval['xgb_pred']
actual = df_cohort_overall_eval['true']
MSE=np.mean((prediction - actual)**2)
MAE=np.mean(np.abs((prediction - actual)))
MAPE=mean_absolute_percentage_error(actual, prediction)
Bias = np.mean(prediction - actual)
print('MAE: ', round(MAE, 2))
print('RMSE: ', round(MSE**0.5, 2))
print('MAPE: ', round(MAPE, 2))
print('Bias: ', round(Bias, 2))

MAE:  360.28
RMSE:  551.92
MAPE:  4.65
Bias:  62.24


In [147]:
df_cohort_hotday_eval = df_cohort_hotday_all.copy()
df_cohort_hotday_eval['timestamp'] = pd.to_datetime(df_cohort_hotday_eval['timestamp'])
df_cohort_hotday_eval = pd.merge(df_cohort_hotday_eval, predicted_values_pivot, how='left')
#print(df_cohort_hotday_eval.columns.values)

prediction = df_cohort_hotday_eval['xgb_pred']

actual = df_cohort_hotday_eval['true']
MSE=np.mean((prediction - actual)**2)
MAE=np.mean(np.abs((prediction - actual)))
MAPE=mean_absolute_percentage_error(actual, prediction)
Bias = np.mean(prediction - actual)
print('MAE: ', round(MAE, 2))
print('RMSE: ', round(MSE**0.5, 2))
print('MAPE: ', round(MAPE, 2))
print('Bias: ', round(Bias, 2))

MAE:  584.35
RMSE:  891.46
MAPE:  6.28
Bias:  -330.77


# Export data

In [150]:
# predictions for df_cohort_overall
df_cohort_overall_all['timestamp'] = pd.to_datetime(df_cohort_overall_all['timestamp'])
df_cohort_overall_all = pd.merge(df_cohort_overall_all, predicted_values_pivot, how='left')
print(df_cohort_overall_all.shape)
df_cohort_overall_all.to_csv(f"{BASE}/xgb/cohort_overall_all_xgb_multiout.csv")

(1854910, 8)


In [151]:
df_cohort_hotday_all['timestamp'] = pd.to_datetime(df_cohort_hotday_all['timestamp'])
df_cohort_hotday_all = pd.merge(df_cohort_hotday_all, predicted_values_pivot, how='left')
print(df_cohort_hotday_all.shape)
df_cohort_hotday_all.to_csv(f"{BASE}/xgb/cohort_hotday_all_xgb_multiout.csv")

(186624, 8)
